# **STEP 1: IMPORT LIBRARIES**

In [ ]:
import pandas as pd # For data manipulation and analysis
from ydata_profiling import ProfileReport
import numpy as np # For numerical operations
import matplotlib.pyplot as plt # For creating visualizations
import seaborn as sns # For statistical data visualization
from sklearn.model_selection import train_test_split,cross_val_score # For splitting data into train/test sets
from sklearn.preprocessing import StandardScaler,LabelEncoder # For encoding and scaling data
from sklearn.impute import SimpleImputer # For handling missing values
from sklearn.linear_model import LogisticRegression # Machine learning model
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier # Machine learning model
import xgboost as xgb # For XGBoost model
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc, confusion_matrix, RocCurveDisplay # For model evaluation
import warnings # To suppress warnings
warnings.filterwarnings('ignore') # To suppress warnings

print("\n Libraries Loaded succesfully...")

# **STEP 2: LOAD AND EXPLORE DATA**

In [ ]:
from pathlib import Path

# The raw dataset is intentionally local until its exact redistribution rights are verified.
_DATA_CANDIDATES = [
    Path.cwd() / "credit_risk_dataset.csv",
    Path.cwd() / "creadit_risk_fds" / "credit_risk_dataset.csv",
    Path.cwd().parent / "credit_risk_dataset.csv",
]
DATA_PATH = next((candidate for candidate in _DATA_CANDIDATES if candidate.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "credit_risk_dataset.csv is not bundled with this repository. Obtain a copy "
        "from the original publisher, verify its license/terms, and place it in "
        "creadit_risk_fds/ before running this notebook."
    )
df = pd.read_csv(DATA_PATH)
print("="*80)
print("Data Loading")
print("="*80)

# Display first few rows to understand data structure
print(f"\n1. First 5 rows of the dataset{df.sample(5)}")
df.head()


In [ ]:
# Get basic information about the dataset
print(f"\n2. Dataset Information:{df.info()}")
# Check the shape (rows and columns)
print(f"\n3. Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# Display statistical summary of numerical columns
print(f"\n4. Statistical Summary:{df.describe()}")

In [ ]:
#Display information about dataframe
df.info()

In [ ]:
# Display column names
print(f"Columns :{list(df.columns)}")
# Check for missing values in each column
print(f"\n5. Missing Values Count:{df.isnull().sum()}")

# **STEP3:Profiling Report**

In [ ]:
print("="*80)
print(" Generating Profiling Report")
print("="*80)
profile = ProfileReport(df, title="Credit Risk Dataset", explorative=True)
profile.to_file("loan_data_profiling_report")
print("\n Profiling report saved to")

In [ ]:
# EDA: Target Distribution
sns.countplot(x='loan_status', data=df)
plt.title("Loan Status Distribution")
plt.show()

In [ ]:
# EDA: Numerical Features Distribution
num_cols = ['person_age','person_income','person_emp_length','loan_amnt',
            'loan_int_rate','loan_percent_income','cb_person_cred_hist_length']
df[num_cols].hist(bins=20, figsize=(15,10))
plt.suptitle("Numerical Features Distribution")
plt.show()

In [ ]:
# EDA: Categorical Features Countplots
cat_cols = ['person_home_ownership','loan_intent','loan_grade','cb_person_default_on_file']

for col in cat_cols:
    plt.figure(figsize=(10,5))
    sns.countplot(x=col, data=df, order=df[col].value_counts().index)
    plt.title(f"{col} Distribution")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# EDA: Correlation Heatmap
plt.figure(figsize=(10,6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation (Numeric Only)")
plt.show()

In [ ]:
# EDA: Feature vs Target
plt.figure(figsize=(8,6))
sns.boxplot(x='loan_status', y='loan_amnt', data=df)
plt.title("Loan Amount vs Loan Status")
plt.show()

plt.figure(figsize=(8,6))
sns.countplot(x='person_home_ownership', hue='loan_status', data=df)
plt.title("Home Ownership vs Loan Status")
plt.show()

# **PART 4: PREPARE DATA FOR MACHINE LEARNING**

In [ ]:
# Handle missing values
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Encode categorical features
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

# Features & target
X = df.drop('loan_status', axis=1)
y = df['loan_status']

# Train/test split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features for Logistic Regression
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])

# **PART 8: BUILD MODEL**

In [ ]:
# Baseline Model: Logistic Regression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)

print("Logistic Regression Performance:")
print(classification_report(y_val, y_pred_lr))

roc_auc_lr = roc_auc_score(y_val, lr.predict_proba(X_val)[:,1])
print(f"ROC AUC Score: {roc_auc_lr:.4f}")

# ROC Curve
RocCurveDisplay.from_estimator(lr, X_val, y_val)
plt.title("ROC Curve - Logistic Regression")
plt.show()

In [ ]:
# Tree-based Model: Random Forest

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)

print("Random Forest Performance:")
print(classification_report(y_val, y_pred_rf))

roc_auc_rf = roc_auc_score(y_val, rf.predict_proba(X_val)[:,1])
print(f"ROC AUC Score: {roc_auc_rf:.4f}")

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=feat_imp, y=feat_imp.index)
plt.title("Feature Importance - Random Forest")
plt.show()

# ROC Curve
RocCurveDisplay.from_estimator(rf, X_val, y_val)
plt.title("ROC Curve - Random Forest")
plt.show()

In [ ]:
# Advanced Model: XGBoost

# Initialize and train model
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_val)
y_score = xgb_model.predict_proba(X_val)[:,1]

# Performance metrics
print("XGBoost Performance:")
print(classification_report(y_val, y_pred_xgb))
roc_auc_xgb = roc_auc_score(y_val, y_score)
print(f"ROC AUC Score: {roc_auc_xgb:.4f}")

# Feature importance (top 10)
feat_imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=feat_imp[:10], y=feat_imp.index[:10])
plt.title("Top 10 Feature Importances - XGBoost")
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_val, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0,1], [0,1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - XGBoost')
plt.legend(loc="lower right")
plt.show()

# **Evalution**

In [ ]:
best_model = xgb_model
y_best = y_pred_xgb

cm = confusion_matrix(y_val, y_best)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - XGBoost")
plt.show()